# Grafico 02 - Trayectorias de superficie natural por macrozona

Este notebook reproduce en Google Colab las 4 imagenes de este grafico (general + Parques + Reservas + Monumentos) y su tabla de soporte en Excel.

**Antes de correr las celdas de abajo**, ten a mano el archivo `naturalidad_data.json` (esta en la carpeta `codigo/` de este grafico, en tu computador o en tu repositorio de GitHub).

Corre las celdas en orden, de arriba hacia abajo.

## 1. Instalar paquetes

In [ ]:
!pip -q install numpy pandas matplotlib openpyxl


## 2. Subir el archivo de datos

In [ ]:
from google.colab import files
print("Sube aqui: naturalidad_data.json")
uploaded = files.upload()


## 3. Generar las 4 imagenes

In [ ]:
"""
Trayectorias de superficie natural por macrozona — general + por tipología
=============================================================================

Gráfico corregido según comentarios de Ángela y de Vale (agosto-2026). Ver
README.txt y METODOLOGIA.docx de esta carpeta para el detalle completo de
qué cambió y por qué.

QUÉ HACE ESTE SCRIPT
--------------------
Genera 4 gráficos de líneas, TODOS con el mismo formato: una fila de 5
paneles (uno por macrozona: Norte, Centro, Centro Sur, Sur, Austral), cada
panel mostrando, para las AP de esa macrozona:
  - Una línea fina por AP, con la evolución de su % de superficie natural
    interna a lo largo de los 7 cortes de año (2000 a 2024).
  - Una línea gruesa negra con el PROMEDIO de esa macrozona.

Los 4 gráficos son:
  1. General          -> las 97 AP juntas (todas las tipologías)      -- YA
     EXISTÍA con este mismo formato; Ángela confirmó que funciona muy bien
     para dar el panorama global y pidió conservarlo tal cual (solo se le
     aplicó la limpieza de diseño y terminología de esta ronda de
     correcciones, sin tocar el análisis).
  2. Parques Nacionales (PN)   -> mismo formato, filtrado a PN
  3. Reservas Nacionales (RN)  -> mismo formato, filtrado a RN
  4. Monumentos Naturales (MN) -> mismo formato, filtrado a MN

Estos 3 últimos son NUEVOS: Vale pidió "desagregar" la tendencia general
por tipología para poder comparar mejor entre categorías, tal como se hizo
con el heatmap (gráfico 01). Filtrar por tipología es un cambio pequeño en
el código: se reduce la lista de AP de cada panel antes de graficar (ver
sección 3 más abajo), la lógica del gráfico es idéntica en los 4 casos.

QUÉ SE ELIMINÓ RESPECTO A LA VERSIÓN ANTERIOR
----------------------------------------------
Existía un 5to gráfico ("03b_trayectorias_por_tipologia.png") que intentaba
mostrar Parques/Reservas/Monumentos en 3 paneles, con las líneas de cada AP
coloreadas según su MACROZONA (para diferenciarlas dentro del panel) más
una leyenda de color aparte. Vale pidió eliminarlo por completo: mezclaba
dos criterios de agrupación distintos en una sola imagen (tipología para
separar en paneles + macrozona para colorear las líneas) y el resultado
era confuso. En su lugar quedan los 3 gráficos nuevos (PN/RN/MN) de esta
lista, cada uno en el MISMO formato que el general (separado por
macrozona, no por color) -- mucho más simple y directo de leer.

DISEÑO: solo título + nombres de ejes + el gráfico -- sin subtítulo ni
notas al pie sobre la imagen (esa explicación va en el README/METODOLOGIA).
Terminología corregida: "superficie natural", no "cobertura natural" ni
"vegetación natural".

ARCHIVO DE ENTRADA (debe estar en esta misma carpeta `codigo/`)
------------------------------------------------------------------
  naturalidad_data.json   -> % de superficie natural por AP, año y
                              distancia (acá se usa la distancia "AP",
                              es decir dentro del polígono de la AP misma,
                              a lo largo de los 7 años).

SALIDA (se guarda en ../imagenes/)
-----------------------------------
  trayectorias_general_por_macrozona.png      -> las 97 AP juntas
  trayectorias_parques_por_macrozona.png      -> solo Parques Nacionales
  trayectorias_reservas_por_macrozona.png     -> solo Reservas Nacionales
  trayectorias_monumentos_por_macrozona.png   -> solo Monumentos Naturales

Para correrlo: python3 trayectorias_macrozona.py
(requiere numpy, matplotlib -- instalar con: pip install numpy matplotlib)

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y naturalidad_data.json no está al
     lado -> variable NATURALIDAD_JSON_PATH, más abajo.
  ...quieres que las imágenes se guarden en otro lugar
     -> variable OUT_DIR, más abajo.
  ...agregas o quitas una macrozona, o cambia su orden/color
     -> MACRO_ORDER se lee del JSON; MACRO_COLORS se define más abajo.
  ...quieres agregar/quitar una categoría de tipología (ej. una 5ta)
     -> lista CATEGORIAS, al final del archivo, y la función tipologia().
  ...cambian los años de la serie (hoy: 2000 a 2024 cada 4 años)
     -> se leen automáticamente de YEARS en el JSON, no hay que tocar nada.
  ...quieres cambiar tamaño de letra, grosor de línea, tamaño de figura,
     etc. (ajustes puramente visuales)
     -> están marcados con "<-- AJUSTE VISUAL" en build_trayectorias().
===========================================================================
"""

import json
import re
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# RUTAS DE ARCHIVOS
# ---------------------------------------------------------------------
BASE_DIR = "/content"  # <-- en Colab, los archivos subidos con files.upload() quedan en /content

# <-- CAMBIAR AQUÍ si le cambiaste el nombre al archivo de datos, o si lo
#     moviste a otra carpeta (en ese caso, reemplaza BASE_DIR por la ruta
#     completa a esa carpeta).
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, "naturalidad_data.json")

# <-- CAMBIAR AQUÍ si quieres que los PNG se guarden en otro lugar (por
#     defecto: una carpeta "imagenes" al lado de esta carpeta "codigo").
OUT_DIR = os.path.join(BASE_DIR, "imagenes")
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------
# PALETA Y ESTILO (igual al resto del proyecto, para consistencia visual)
# ------------------------------------------------------------------
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"

# <-- CAMBIAR AQUÍ el color de cada macrozona (mismas 5 llaves que
#     MACRO_ORDER, que se lee del JSON más abajo).
MACRO_COLORS = {
    "Norte": "#eda100", "Centro": "#1baf7a", "Centro Sur": "#4a3aa7",
    "Sur": "#2a78d6", "Austral": "#e34948",
}

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]


def style_ax(ax):
    """Estilo visual estándar del proyecto: quita bordes superior/derecho,
    colorea ticks y etiquetas."""
    ax.set_facecolor(SURFACE)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color(BASELINE)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    ax.xaxis.label.set_color(INK_SECONDARY)
    ax.yaxis.label.set_color(INK_SECONDARY)


# ------------------------------------------------------------------
# 1. CARGA DE DATOS
# ------------------------------------------------------------------
# naturalidad_data.json trae, para cada AP: nombre, macrozona ("macro"),
# región, superficie en hectáreas, y el % de superficie natural por año
# ("anillos"[AP][año][índice de distancia]). Acá se usa la distancia "AP"
# (índice 0 dentro de "dist"), es decir el % natural DENTRO del polígono
# de cada área protegida, no en los anillos exteriores.
DATA = json.load(open(NATURALIDAD_JSON_PATH))
YEARS = DATA["years"]            # ej. [2000, 2004, 2008, ..., 2024]
DIST = DATA["dist"]              # ej. ["AP", "1km", ..., "10km"]
MACRO_ORDER = DATA["macroOrder"]  # orden oficial de macrozonas, ya viene en el JSON
APS = DATA["aps"]
ANILLOS = DATA["anillos"]

# Etiqueta corta de cada año para el eje X (ej. 2000 -> "'00"), para que
# quepan los 7 años sin que se amontonen los números.
YEAR_SHORT = {y: f"'{str(y)[-2:]}" for y in YEARS}


def tipologia(nombre):
    """
    Deriva la tipología (PN/RN/MN) del PREFIJO del nombre de la AP
    (ej. "RN Los Queules" -> "RN"). No hay una columna separada para esto
    en los datos originales.
    <-- CAMBIAR AQUÍ si en el futuro los nombres de AP vienen con otro
        formato (ej. la tipología al final del nombre, o en minúsculas).
    """
    m = re.match(r"^(MN|PN|RN)\s", nombre)
    return m.group(1) if m else "??"


for a in APS:
    a["tipo"] = tipologia(a["name"])


def ap_val(ap_name, year, dist_label):
    """% de superficie natural de una AP, en un año y distancia dados."""
    idx = DIST.index(dist_label)
    return ANILLOS[ap_name][str(year)][idx]


# ------------------------------------------------------------------
# 2. FUNCIÓN QUE ARMA EL GRÁFICO DE 5 PANELES (uno por macrozona)
#    filtrado por tipología (o con todas las AP si tipo_filter=None).
#
#    tipo_filter : None (todas las AP) | "PN" | "RN" | "MN"
#    titulo_tipo : texto que se muestra como 2da línea del título
#    out_name    : nombre del archivo .png de salida
# ------------------------------------------------------------------
def build_trayectorias(tipo_filter, titulo_tipo, out_name):
    subset = [a for a in APS if tipo_filter is None or a["tipo"] == tipo_filter]

    fig, axes = plt.subplots(1, 5, figsize=(13, 4.5), dpi=200, sharey=True)
    # <-- AJUSTE VISUAL: figsize (tamaño de la imagen); sharey=True hace
    #     que los 5 paneles compartan la misma escala vertical (0-100%),
    #     para que sean directamente comparables entre sí.
    fig.patch.set_facecolor(SURFACE)

    for ax, macro in zip(axes, MACRO_ORDER):
        style_ax(ax)
        aps_in_macro = [a for a in subset if a["macro"] == macro]
        all_vals = []
        for a in aps_in_macro:
            ys = [ap_val(a["name"], y, "AP") for y in YEARS]
            if any(v is None for v in ys):
                # AP sin dato completo en algún año -- se omite su línea
                # individual (pero no rompe el promedio de las demás).
                continue
            ax.plot(YEARS, ys, color=MACRO_COLORS[macro], linewidth=0.9, alpha=0.35, zorder=2)
            # <-- AJUSTE VISUAL: linewidth=0.9 y alpha=0.35 (grosor y
            #     transparencia de cada línea individual de AP)
            all_vals.append(ys)
        if all_vals:
            mean_vals = np.nanmean(np.array(all_vals, dtype=float), axis=0)
            ax.plot(YEARS, mean_vals, color=INK_PRIMARY, linewidth=2.2, zorder=4)
            # <-- AJUSTE VISUAL: linewidth=2.2 (grosor de la línea de
            #     promedio, la que resalta en negro sobre las demás)
        ax.set_title(f"{macro}\n({len(aps_in_macro)} AP)", color=INK_PRIMARY,
                     fontsize=10.5, fontweight="bold", pad=8)
        ax.set_xticks(YEARS)
        ax.set_xticklabels([YEAR_SHORT[y] for y in YEARS], fontsize=6.6, rotation=0)
        ax.set_xlabel("Año", fontsize=9)
        ax.set_ylim(-2, 102)  # <-- AJUSTE VISUAL: rango del eje Y (0-100% con un poco de margen)
        # Fija el rango del eje X explícitamente a partir de los años reales
        # (en vez de dejar que matplotlib lo calcule solo a partir de los
        # datos graficados). Esto es necesario para los paneles filtrados
        # por tipología: cuando una macrozona no tiene NINGUNA AP de esa
        # tipología (ej. "Centro Sur" no tiene Monumentos Naturales), el
        # panel queda vacío (sin líneas) y, sin este set_xlim(), los 7
        # ticks de año colapsan todos en el mismo punto y se ven amontonados
        # unos sobre otros -- un bug que se detectó y corrigió en esta
        # misma ronda de correcciones al revisar visualmente el resultado.
        ax.set_xlim(YEARS[0] - (YEARS[1] - YEARS[0]) * 0.3, YEARS[-1] + (YEARS[1] - YEARS[0]) * 0.3)
        ax.grid(axis="y", color=GRID, linewidth=0.7, zorder=0)
        ax.set_axisbelow(True)
        if not aps_in_macro:
            # Caso borde: esta macrozona no tiene ninguna AP de la
            # tipología filtrada (ej. Centro Sur no tiene Monumentos
            # Naturales). En vez de dejar el panel completamente vacío
            # (que se puede confundir con un error), se avisa con texto.
            ax.text(0.5, 0.5, "Sin AP de esta\ntipología en esta\nmacrozona",
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=8, color=INK_MUTED, style="italic", linespacing=1.4)
            # <-- AJUSTE VISUAL: fontsize=8, texto del aviso "sin AP"

    axes[0].set_ylabel("% superficie natural dentro de la AP")

    # Solo título + nombre de la categoría (sin subtítulo/nota al pie
    # debajo -- únicamente título, nombres de ejes y el gráfico mismo).
    fig.suptitle(f"Evolución de la superficie natural dentro de las AP, por macrozona\n{titulo_tipo}",
                 color=INK_PRIMARY, fontsize=14.5, fontweight="bold", x=0.02, ha="left", y=0.995, va="top")
                 # <-- AJUSTE VISUAL: fontsize=14.5 y posición (x, y) del título
    fig.subplots_adjust(top=0.78, bottom=0.14, left=0.055, right=0.98, wspace=0.12)
    # <-- AJUSTE VISUAL: estos 5 números controlan los márgenes de la
    #     figura y la separación entre los 5 paneles (wspace).

    fig.savefig(os.path.join(OUT_DIR, out_name), facecolor=SURFACE)
    plt.close(fig)
    print(f"OK {out_name} ({len(subset)} AP)")


# ------------------------------------------------------------------
# 3. GENERAR LOS 4 GRÁFICOS
#
# Cada tupla es: (filtro de tipología, texto que aparece en el título,
# nombre del archivo de salida).
#
# <-- CAMBIAR AQUÍ para agregar/quitar una categoría, o cambiar el nombre
#     de un archivo de salida. El filtro de tipología debe ser None (todas
#     las AP), o coincidir exactamente con lo que devuelve la función
#     tipologia() más arriba (ej. "PN", "RN", "MN").
# ------------------------------------------------------------------
CATEGORIAS = [
    (None, "General", "trayectorias_general_por_macrozona.png"),
    ("PN", "Parques Nacionales (PN)", "trayectorias_parques_por_macrozona.png"),
    ("RN", "Reservas Nacionales (RN)", "trayectorias_reservas_por_macrozona.png"),
    ("MN", "Monumentos Naturales (MN)", "trayectorias_monumentos_por_macrozona.png"),
]

for tipo_filter, titulo_tipo, out_name in CATEGORIAS:
    build_trayectorias(tipo_filter, titulo_tipo, out_name)


## 4. Generar la tabla de soporte (Excel)

In [ ]:
"""
Tabla de soporte del gráfico 03 (trayectorias de superficie natural por macrozona)
=====================================================================================

Genera tabla_soporte.xlsx (en la carpeta de arriba, junto a README.txt) con
los datos exactos que usan los 4 gráficos de trayectorias, en 2 hojas:

  1. trayectorias_detalle  -> una fila por AP, con su % de superficie
                               natural en cada uno de los 7 años (2000 a
                               2024) y el cambio máximo entre 2 cortes
                               consecutivos (para detectar caídas o
                               recuperaciones abruptas).
  2. promedio_por_macrozona -> el promedio que se ve como línea gruesa
                               negra en cada panel, para las 4 versiones
                               (general/PN/RN/MN) en una sola tabla larga.

Requiere: numpy, pandas, openpyxl
Para correrlo: python3 tabla_soporte.py

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y naturalidad_data.json no está al
     lado -> variable NATURALIDAD_JSON_PATH, más abajo.
  ...quieres que tabla_soporte.xlsx se guarde en otro lugar
     -> variable OUT_XLSX, más abajo.
  ...quieres agregar una hoja nueva -> agregar un DataFrame más y una
     línea más en el bloque "Guardar todo en un xlsx", al final.
===========================================================================
"""

import json
import re
import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# RUTAS DE ARCHIVOS
# ---------------------------------------------------------------------
BASE_DIR = "/content"  # <-- en Colab, los archivos subidos con files.upload() quedan en /content

# <-- CAMBIAR AQUÍ si le cambiaste el nombre al archivo de datos, o si lo
#     moviste a otra carpeta.
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, "naturalidad_data.json")

# <-- CAMBIAR AQUÍ si quieres que tabla_soporte.xlsx se guarde en otro
#     lugar (por defecto: un nivel arriba de esta carpeta "codigo").
OUT_XLSX = os.path.join(BASE_DIR, "tabla_soporte.xlsx")

DATA = json.load(open(NATURALIDAD_JSON_PATH))
YEARS = DATA["years"]
DIST = DATA["dist"]
MACRO_ORDER = DATA["macroOrder"]
APS = DATA["aps"]
ANILLOS = DATA["anillos"]


def tipologia(nombre):
    """PN/RN/MN a partir del prefijo del nombre de la AP.
    <-- CAMBIAR AQUÍ si el formato de los nombres cambia en el futuro."""
    m = re.match(r"^(MN|PN|RN)\s", nombre)
    return m.group(1) if m else "??"


TIPO_NOMBRE = {"PN": "Parque Nacional", "RN": "Reserva Nacional", "MN": "Monumento Natural"}

for a in APS:
    a["tipo"] = tipologia(a["name"])


def ap_val(ap_name, year, dist_label):
    idx = DIST.index(dist_label)
    return ANILLOS[ap_name][str(year)][idx]


# ---------------------------------------------------------------
# 1) HOJA trayectorias_detalle: una fila por AP
# ---------------------------------------------------------------
rows = []
for a in APS:
    nm = a["name"]
    vals = [ap_val(nm, y, "AP") for y in YEARS]
    row = {"AP": nm, "tipologia": a["tipo"], "macrozona": a["macro"], "region": a["region"]}
    for y, v in zip(YEARS, vals):
        row[f"pct_natural_{y}"] = v
    if all(v is not None for v in vals):
        # cambio_maximo_entre_cortes_pp: la mayor variación (en valor
        # absoluto) entre 2 cortes de año CONSECUTIVOS, y en qué período
        # ocurrió -- sirve para detectar caídas o recuperaciones abruptas
        # que no se ven a simple vista en el gráfico de líneas.
        deltas = [vals[i + 1] - vals[i] for i in range(len(vals) - 1)]
        i_max = int(np.argmax(np.abs(deltas)))
        row["cambio_maximo_entre_cortes_pp"] = round(deltas[i_max], 2)
        row["periodo_del_cambio_maximo"] = f"{YEARS[i_max]}–{YEARS[i_max + 1]}"
    else:
        row["cambio_maximo_entre_cortes_pp"] = None
        row["periodo_del_cambio_maximo"] = None
    rows.append(row)
df_tray = pd.DataFrame(rows)
# ordenado por magnitud del cambio máximo (mayor variación abrupta primero)
df_tray = df_tray.reindex(df_tray["cambio_maximo_entre_cortes_pp"].abs().sort_values(ascending=False).index)

# ---------------------------------------------------------------
# 2) HOJA promedio_por_macrozona: la línea negra de cada panel,
#    para las 4 versiones del gráfico en una tabla larga (columna
#    "version" indica a cuál corresponde cada fila).
# ---------------------------------------------------------------
CATEGORIAS = [(None, "general"), ("PN", "parques"), ("RN", "reservas"), ("MN", "monumentos")]
prom_rows = []
for tipo_filter, version in CATEGORIAS:
    subset = [a for a in APS if tipo_filter is None or a["tipo"] == tipo_filter]
    for m in MACRO_ORDER:
        aps_m = [a["name"] for a in subset if a["macro"] == m]
        row = {"version": version, "macrozona": m, "n_ap": len(aps_m)}
        if aps_m:
            for y in YEARS:
                vals = [ap_val(nm, y, "AP") for nm in aps_m]
                vals = [v for v in vals if v is not None]
                row[f"promedio_{y}"] = round(float(np.mean(vals)), 2) if vals else None
        else:
            for y in YEARS:
                row[f"promedio_{y}"] = None
        prom_rows.append(row)
df_prom = pd.DataFrame(prom_rows)

# ---------------------------------------------------------------
# Guardar todo en un xlsx con 2 hojas
# ---------------------------------------------------------------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_tray.to_excel(writer, sheet_name="trayectorias_detalle", index=False)
    df_prom.to_excel(writer, sheet_name="promedio_por_macrozona", index=False)

print(f"OK {OUT_XLSX} -- {len(df_tray)} AP, 2 hojas")


## 5. Ver las imagenes generadas

In [ ]:
import glob
from IPython.display import Image, display

for p in sorted(glob.glob(os.path.join(OUT_DIR, '*.png'))):
    print(p.split('/')[-1])
    display(Image(filename=p))


## 6. Descargar todo (imagenes + tabla de soporte) en un .zip

In [ ]:
import shutil, os
from google.colab import files

RESULT_DIR = "/content/resultados_02_trayectorias"
os.makedirs(RESULT_DIR, exist_ok=True)
if os.path.isdir(OUT_DIR):
    shutil.copytree(OUT_DIR, os.path.join(RESULT_DIR, "imagenes"), dirs_exist_ok=True)
if os.path.exists(OUT_XLSX):
    shutil.copy(OUT_XLSX, RESULT_DIR)
shutil.make_archive(RESULT_DIR, "zip", RESULT_DIR)
files.download(RESULT_DIR + ".zip")
print("Listo:", RESULT_DIR + ".zip")
